# IRIS ML Pipeline on Vertex AI — Week 1 (Driver Notebook)

**Roll No:** 23f2004634 · **Branch:** `week_1` · **Term:** MAY 2026

This notebook is a thin driver that runs the pipeline scripts end-to-end so the
whole flow can be demonstrated in one top-to-bottom execution. The real logic
lives in the standalone, reproducible scripts:

- `data_prep.py` — split + upload data to GCS (Task 2)
- `train.py` — fetch data, train, store timestamped artifacts (Task 3)
- `inference.py` — fetch model from GCS, run on eval set (Task 4)


## 0. Environment
Set the project and bucket. These feed `config.py` via environment variables.

In [1]:
import os
os.environ["PROJECT_ID"] = os.popen("gcloud config get-value project 2>/dev/null").read().strip()
os.environ["BUCKET"] = "23f2004634-mlops-week1"
print("PROJECT_ID:", os.environ["PROJECT_ID"])
print("BUCKET    :", os.environ["BUCKET"])

PROJECT_ID: project-a0a1f4bf-9c68-4b93-a84
BUCKET    : 23f2004634-mlops-week1


Confirm the starter data is present (cloned from the `ga_resources` repo, branch `week_1`).

In [2]:
!ls ga_resources/data/raw ga_resources/data/v1 ga_resources/data/v2

ga_resources/data/raw:
iris.csv

ga_resources/data/v1:
data.csv

ga_resources/data/v2:
data.csv


## Task 2 — Store training data in GCS
Stratified train/eval split, uploaded to `gs://<bucket>/data/raw/`.

In [3]:
!python data_prep.py --src ga_resources/data/raw/iris.csv --version raw

Uploading split for version 'raw' to gs://23f2004634-mlops-week1/data/raw/
  uploaded -> gs://23f2004634-mlops-week1/data/raw/train.csv  (90 rows)
  uploaded -> gs://23f2004634-mlops-week1/data/raw/eval.csv  (60 rows)
Done.


## Task 3 — Train (Run #1)
Fetch data from GCS, train, store artifacts in a **timestamped** folder.

In [4]:
!python train.py --version raw

Run timestamp: 2026-06-15T09-28-31
Train accuracy: 0.978
  uploaded -> gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-31/model.joblib
  uploaded -> gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-31/metrics.json
  uploaded -> gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-31/metadata.json
  uploaded -> gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-31/training.log
Artifacts stored at gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-31/


## Task 4 — Inference (Run #1)
Fetch the latest model from GCS and run on the eval set.

In [5]:
!python inference.py --version raw

Loading model from gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-31/model.joblib
  downloaded <- gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-31/model.joblib
Eval accuracy: 0.983
  uploaded -> gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-31/inference/predictions.csv  (60 rows)
  uploaded -> gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-31/inference/eval_metrics.json
Inference outputs stored at gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-31/inference/


## Task 5 — Second full run
Produces a **second** timestamped folder, proving traceable, repeatable runs.

In [6]:
!python train.py --version raw && python inference.py --version raw

Run timestamp: 2026-06-15T09-28-44
Train accuracy: 0.978
  uploaded -> gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-44/model.joblib
  uploaded -> gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-44/metrics.json
  uploaded -> gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-44/metadata.json
  uploaded -> gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-44/training.log
Artifacts stored at gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-44/
Loading model from gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-44/model.joblib
  downloaded <- gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-44/model.joblib
Eval accuracy: 0.983
  uploaded -> gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-44/inference/predictions.csv  (60 rows)
  uploaded -> gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-44/inference/eval_metrics.json
Inference outputs stored at gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15

### Verify: two timestamped run folders exist for `raw`

In [7]:
!gsutil ls gs://$BUCKET/artifacts/raw/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-32-30/
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-33-04/
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-31/
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-15T09-28-44/


## Task 6 (Optional) — Multiple data versions
Run the same pipeline on two different data versions and compare.

In [8]:
!python data_prep.py --src ga_resources/data/v1/data.csv --version v1
!python train.py --version v1 && python inference.py --version v1

Uploading split for version 'v1' to gs://23f2004634-mlops-week1/data/v1/
  uploaded -> gs://23f2004634-mlops-week1/data/v1/train.csv  (60 rows)
  uploaded -> gs://23f2004634-mlops-week1/data/v1/eval.csv  (41 rows)
Done.
Run timestamp: 2026-06-15T09-29-04
Train accuracy: 0.950
  uploaded -> gs://23f2004634-mlops-week1/artifacts/v1/2026-06-15T09-29-04/model.joblib
  uploaded -> gs://23f2004634-mlops-week1/artifacts/v1/2026-06-15T09-29-04/metrics.json
  uploaded -> gs://23f2004634-mlops-week1/artifacts/v1/2026-06-15T09-29-04/metadata.json
  uploaded -> gs://23f2004634-mlops-week1/artifacts/v1/2026-06-15T09-29-04/training.log
Artifacts stored at gs://23f2004634-mlops-week1/artifacts/v1/2026-06-15T09-29-04/
Loading model from gs://23f2004634-mlops-week1/artifacts/v1/2026-06-15T09-29-04/model.joblib
  downloaded <- gs://23f2004634-mlops-week1/artifacts/v1/2026-06-15T09-29-04/model.joblib
Eval accuracy: 0.951
  uploaded -> gs://23f2004634-mlops-week1/artifacts/v1/2026-06-15T09-29-04/inference

In [9]:
!python data_prep.py --src ga_resources/data/v2/data.csv --version v2
!python train.py --version v2 && python inference.py --version v2

Uploading split for version 'v2' to gs://23f2004634-mlops-week1/data/v2/
  uploaded -> gs://23f2004634-mlops-week1/data/v2/train.csv  (29 rows)
  uploaded -> gs://23f2004634-mlops-week1/data/v2/eval.csv  (20 rows)
Done.
Run timestamp: 2026-06-15T09-29-20
Train accuracy: 1.000
  uploaded -> gs://23f2004634-mlops-week1/artifacts/v2/2026-06-15T09-29-20/model.joblib
  uploaded -> gs://23f2004634-mlops-week1/artifacts/v2/2026-06-15T09-29-20/metrics.json
  uploaded -> gs://23f2004634-mlops-week1/artifacts/v2/2026-06-15T09-29-20/metadata.json
  uploaded -> gs://23f2004634-mlops-week1/artifacts/v2/2026-06-15T09-29-20/training.log
Artifacts stored at gs://23f2004634-mlops-week1/artifacts/v2/2026-06-15T09-29-20/
Loading model from gs://23f2004634-mlops-week1/artifacts/v2/2026-06-15T09-29-20/model.joblib
  downloaded <- gs://23f2004634-mlops-week1/artifacts/v2/2026-06-15T09-29-20/model.joblib
Eval accuracy: 1.000
  uploaded -> gs://23f2004634-mlops-week1/artifacts/v2/2026-06-15T09-29-20/inference

### Compare eval accuracy across data versions

In [10]:
import json
from google.cloud import storage
client = storage.Client()
bucket = os.environ["BUCKET"]

print(f"{'version':<8}{'run':<22}{'eval_acc':>9}")
print("-"*40)
for v in ["raw", "v1", "v2"]:
    # latest run for this version
    prefixes = set()
    it = client.list_blobs(bucket, prefix=f"artifacts/{v}/", delimiter="/")
    list(it)
    runs = sorted(p.rstrip('/').split('/')[-1] for p in it.prefixes)
    if not runs:
        continue
    run = runs[-1]
    blob = client.bucket(bucket).blob(f"artifacts/{v}/{run}/inference/eval_metrics.json")
    m = json.loads(blob.download_as_text())
    print(f"{v:<8}{run:<22}{m['eval_accuracy']:>9.3f}")

version run                    eval_acc
----------------------------------------
raw     2026-06-15T09-28-44       0.983
v1      2026-06-15T09-29-04       0.951
v2      2026-06-15T09-29-20       1.000


### Final artifact tree
The full organized output: each version → each timestamped run → model, logs, metrics, and inference outputs.

In [11]:
!gsutil ls -r gs://$BUCKET/artifacts/ | head -60

gs://23f2004634-mlops-week1/artifacts/:

gs://23f2004634-mlops-week1/artifacts/raw/:

gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-32-30/:
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-32-30/metadata.json
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-32-30/metrics.json
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-32-30/model.joblib
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-32-30/training.log

gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-32-30/inference/:
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-32-30/inference/eval_metrics.json
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-32-30/inference/predictions.csv

gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-33-04/:
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-33-04/metadata.json
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-33-04/metrics.json
gs://23f2004634-mlops-week1/artifacts/raw/2026-06-14T21-33-04/model.joblib
gs://

---
**Summary:** data fetched from GCS, model trained and stored as timestamped
artifacts, inference run from GCS-fetched models, repeated twice, and compared
across data versions — all on GCP / Vertex AI Workbench.